# Diffusion Policy training on Kaggle (SO-101 hand_tracking_pick_place)

**Before running — Notebook settings (right panel):**
- **Accelerator:** GPU **T4 x2** (use T4, **not P100** — the current Kaggle torch 2.10 dropped support for P100's sm_60; you'll see "not compatible with the current PyTorch installation" and training fails).
- **Internet:** **On** (needed for pip + HF dataset/push)
- **Add-ons -> Secrets:** add a secret named **`HF_TOKEN`** = your Hugging Face *write* token

Workflow: run cells 1-4 interactively to validate (STEPS=2000), check the it/s, then set STEPS=100000 and use **Save Version -> Save & Run All (Commit)** to run headless to completion.

In [ ]:
# 1. Install LeRobot, torchcodec and the grip-aux policy plugin.
!pip install -q "lerobot[diffusion,training] @ git+https://github.com/huggingface/lerobot@da92db8fc0c935950a56b1ea61fa9b211ef3ac30"
# torchcodec fixes the AV1 decode bottleneck. If it errors, comment the next line out and
# remove `--dataset.video_backend=torchcodec` below (falls back to the default decoder).
!pip install -q torchcodec
!pip install -q "huggingface_hub==1.19.0" "transformers==5.5.4"

# Upload webcam-input/.../lerobot_policy_grip_aux as a Kaggle Dataset and attach it as an Input.
import glob, os
_plugin = glob.glob("/kaggle/input/**/lerobot_policy_grip_aux/pyproject.toml", recursive=True)
if not _plugin:
    raise RuntimeError("Attach a Kaggle input containing the lerobot_policy_grip_aux source directory")
PLUGIN_ROOT = os.path.dirname(_plugin[0])
!pip install -q --no-deps --no-build-isolation {PLUGIN_ROOT}
print("grip-aux plugin:", PLUGIN_ROOT)

In [ ]:
# 2. Log in to Hugging Face from the Kaggle secret
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# 3. Sanity check GPU + torchcodec
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))
try:
    import torchcodec; print("torchcodec OK", torchcodec.__version__)
except Exception as e:
    print("torchcodec NOT available -> remove the --dataset.video_backend flag below.", e)

In [ ]:
# 4. Train.  ====== EDIT THESE THREE KNOBS ======
STEPS         = 2000     # 2000 = validation/speed check. Real run: ~90000 (1 GPU) or ~45000 (2 GPUs).
FAST_HORIZON  = True     # True = horizon 16 / n_action_steps 8 (fast, standard DP). False = keep 64/32.
USE_BOTH_GPUS = False    # True = use Kaggle's T4 x2 via accelerate (~2x faster). Requires Accelerator: GPU T4 x2.
BATCH         = 32       # per-GPU batch. Lower to 24/16 if you still hit CUDA out-of-memory.
# =================================================
#
# The policy still predicts the existing 6D action. PV only supervises a masked auxiliary head.
PUSH = (STEPS >= 45000)  # only push a real (long) run to the Hub

# Shared training args (NOT including the launcher or the AMP flag, which differ per path).
args = (
    " --policy.type=grip_aux_diffusion"
    " --dataset.repo_id=stevenzenith/hand_tracking_pv_pick_place"
    " --dataset.video_backend=torchcodec"
    f" --batch_size={BATCH}"     # per-GPU batch (global = BATCH x num_gpus)
    " --num_workers=4"
    " --policy.grip_aux_weight=0.25"
    + (" --policy.horizon=16 --policy.n_action_steps=8" if FAST_HORIZON else "")
    + f" --steps={STEPS}"
    " --save_freq=10000"
    " --output_dir=/kaggle/working/dp_pv_grip"
    " --job_name=dp_pv_grip"
    " --policy.device=cuda"
    f" --policy.push_to_hub={str(PUSH).lower()}"
    " --policy.repo_id=stevenzenith/dp_pv_grip"
    " --wandb.enable=false"
)

# expandable_segments avoids allocator fragmentation (AMP mixes fp16/fp32 buffers -> the
# "reserved but unallocated" OOM on the 15GB T4). Lets PyTorch reuse freed segments.
ALLOC = "PYTORCH_ALLOC_CONF=expandable_segments:True "
if USE_BOTH_GPUS:
    cmd = ALLOC + "accelerate launch --multi_gpu --num_processes=2 --mixed_precision=fp16 $(which lerobot-train)" + args
else:
    cmd = ALLOC + "lerobot-train --policy.use_amp=true" + args

print(cmd)
!{cmd}

## Validation and promotion
- The previous DP speed measurements came from the old dataset/policy and are not evidence for this run.
- First run 2000 steps and confirm non-zero `grip_aux_valid`, finite `action_loss`, and decreasing `grip_aux_mae`.
- Evaluate the head on object-held-out episodes against a context-only mean baseline. Keep deployment in
  `--grip-control direct` until the head wins and the original 6D action validation has not regressed.
- Install the same plugin on the robot machine before loading `stevenzenith/dp_pv_grip`; PV itself is not
  supplied to the policy at deployment.